# 1. Sequence Generation
## Multi-Scale Sequence Tokenization
This notebook demonstrates multi-scale nucleotide and BPE/k-mer tokenization strategies ($\mathcal{T}_N$) for genomic RNA sequences, transforming raw nucleotide alphabet strings into subword token IDs prior to GLM embedding initialization.

## GLM Workflow

# Fragmenting long DNA sequences into smaller chunks, tokenizing them as 3-mers, and then performing stopword removal.
DNA Sequence = "ATGCGTACGTAGCTAGGCTA"
Chunks = [ATGCGT,ACGTAG,CTAGGC,TAGCTA]
3-mers = {[ATG,TGC,GCG,CGT],[ACG,CGT,GTA,TAG],[CTA,TAG,AGG,GGC],[TAG,AGC,GCT,CTA]}
Data Cleaning = {[ATG,TGC,GCG,][ACG,GTA][CTA,AGG,GGC][AGC,GCT,CTA]}
Final k-mers : ['ATG','TGC','GCG','ACG','GTA','CTA','AGG','GGC','AGC','GCT','CTA']

# Transformation of Tokens
0. TF-IDF
1. Embedding Layer
2. Word2Vec
3. Transformers: BERT and GPT

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import ByteLevel

tok = Tokenizer(BPE())
tok.pre_tokenizer = Whitespace()                 # ✅ thread-safe on 3.14t
tok.post_processor = ByteLevel(trim_offsets=True)

In [ ]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(["./path/to/dataset/1.txt",
     "./path/to/dataset/2.txt",
     "./path/to/dataset/3.txt",
     "./path/to/sequences.txt"], vocab_size=50265, min_frequency=2, trainer=trainer)
tokenizer.save_model(".", "bio-gpt-tokenizer")

In [ ]:
# Import BioBERT tokenizer
from tokenizers import Tokenizer
if TOKENIZER_PATH.exists():
    tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
    
    # Test sample nucleotide sequence
    sample_seq = 'ACCGTTAGGCTACGATCGGATCCG'
    encoded = tokenizer.encode(sample_seq)
    
    print(f'Sample Sequence: {sample_seq}')
    print(f'Encoded Tokens: {encoded.tokens}')
    print(f'Token IDs: {encoded.ids}')
else:
    print('Tokenizer file not found. Ensure workflow 05 has been executed to train the tokenizer.')

In [ ]:
## Loading Model Checkpoints and Inspecting Embeddings
if MODEL_PATH.exists() and TOKENIZER_PATH.exists():
    model = BertForMaskedLM.from_pretrained(str(MODEL_PATH))
    model.eval()
    
    print('Model Architecture Configuration:')
    print(model.config)
    
    # Inspect embedding layer weights (E_CBOW + P)
    embeddings = model.bert.embeddings.word_embeddings.weight
    print(f'Word embedding matrix shape: {embeddings.shape}')
else:
    print('GLM model checkpoint or tokenizer not found. Run workflow 05 first to train the GLM.')

# 02. Protein Language Model (PLM) Embeddings
This notebook loads the generative Protein Language Model (ProGen2) and extracts contextual sequence representation embeddings ($\mathbf{z}_q^P = f_{\theta_P}(\mathcal{T}_P(\mathbf{a}_q^P))$) for protein sequences as specified in Section 5.2 and Section 9 of the Mathematical Model.

# 03. Exerkine Identification
This notebook calculates the exercise-response probability score ($\rho_q = P(E_q = 1 \mid \mathbf{z}_q, \mathbf{X}, \mathbf{Y})$) and defines the predicted exerkine set ($E$) as specified in Section 5.4 of the Mathematical Model.

## Loading and Inspecting Identified Exerkines Set ($E$)

In [ ]:
exerkines_path = EXERKINES_DIR / 'identified_exerkines.csv'
if exerkines_path.exists():
    df = pd.read_csv(exerkines_path)
    print(f'Total identified exerkines in set E: {len(df)}')
    display(df.head(10))
else:
    print('Exerkine identity files not found. Ensure workflow 07 has been executed.')

## Data Exploration: Inspecting Raw AnnData (.h5ad) Objects
## Single Cell Annotation: 
# 04. Single-Cell Quality Control & Annotation
This notebook processes single-cell transcriptomics data ($\mathbf{X}$) for the EXERKINEMAP framework, performing quality filtering, normalization, highly variable gene extraction, and UMAP dimensionality reduction.



In [ ]:
# Scanpy analysis
# Plot UMAP colored by cell type or clusters if available
# color_key = 'cell_type' if 'cell_type' in adata.obs else adata.obs_keys()[0]
# sc.pl.umap(adata, color=[color_key], title='EXERKINEMAP Cellular UMAP Embedding')

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

# Set up plotting parameters
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# 05. Ligand-Receptor Interaction Network
This notebook constructs the base ligand-receptor interaction network ($\mathcal{LR}$) integrating BioGRID, STRING, and LIANA+ biological priors, along with molecular compatibility scores ($\Gamma_{km}$) and biological prior weights ($\alpha_{km}$) as defined in Section 10 of the [EXERKINEMAP Mathematical Model](https://github.com/gomezdj/exerkinemap/blob/main/mathematical_model.md).

## Loading and Inspecting the Base LR Network

In [ ]:
lr_path = LR_DIR / 'exerkine_lr_network.csv'
if lr_path.exists():
    lr_df = pd.read_csv(lr_path)
    print(f'Loaded interaction network with {len(lr_df)} edges.')
    display(lr_df.head(10))
else:
    print('Ligand-receptor network file not found. Ensure workflow script 08 has been executed.')

# 06. Spatial Communication Network
This notebook computes the physical spatial kernel ($\mathcal{K}_{ij}^S$) based on tissue coordinates and constructs the spatially informed interaction score ($\widetilde{S}_{ij}^{(l_k,r_m)}$) as defined in Section 10 of the [EXERKINEMAP Mathematical Model](https://github.com/gomezdj/exerkinemap/blob/main/mathematical_model.md).

## Loading Spatial Datasets & Computing Spatial Kernel ($\mathcal{K}_{ij}^S$)

In [ ]:
spatial_path = PROCESSED_SPATIAL_DIR / 'motrpac_spatial_processed.h5ad'
if spatial_path.exists():
    adata = sc.read_h5ad(spatial_path)
    coords = adata.obsm['spatial']
    
    # Compute Gaussian distance decay kernel: K_ij^S = exp(-|s_i - s_j|^2 / (2 * sigma_S^2))
    sigma_S = 100.0
    dist_sq = cdist(coords, coords, metric='sqeuclidean')
    K_S = np.exp(-dist_sq / (2 * (sigma_S ** 2)))
    
    print(f'Spatial Kernel computed. Shape: {K_S.shape}')
    print(f'Non-zero spatial edges: {np.count_nonzero(K_S)}')
else:
    print('Processed spatial dataset not found. Ensure workflow script 03 and 09 have been executed.')

# 07. Signal Propagation and ELSA Scaling

This notebook constructs the sparse Graph Laplacian ($\mathcal{L}_E = \mathbf{D} - \mathbf{W}_E$) and simulates the spatial propagation of exerkine signals ($\mathbf{F}(t) = e^{-t\mathcal{L}_E}\mathbf{f}_0$) across tissue microenvironments, per the [EXERKINEMAP Mathematical Model](https://github.com/gomezdj/exerkinemap/blob/main/mathematical_model.md).

The propagated signal is then modulated using Entropy-based Local Indicators of Spatial Association (ELSA). ELSA computes category distributions within local neighborhoods to decompose entropy into cell- or tile-level values, revealing micro-regional clusters, gradients, and transitions. 

By utilizing an $m$-nearest neighbor graph (which scales efficiently with $O(Nm)$ complexity), ELSA scores ($\mathbf{E}_{local}$) act as a scaling factor to highlight heterogeneity hotspots—such as immune-cell mixing or sharp tissue boundaries—that global metrics miss. 

**Algorithmic Note:** Resolution depends on neighborhood size $m$. A small $m$ may miss broader patterns, while a large $m$ can dilute local signals.

The spatially-aware propagation algorithm is defined as:

$$ \mathbf{F}_{ELSA}(t) = \left( e^{-t \mathcal{L}_E} \mathbf{f}_0 \right) \odot \mathbf{E}_{local} $$

In [ ]:
import numpy as np
import scanpy as sc
from scipy.sparse.linalg import expm_multiply
from scipy.spatial import cKDTree
from scipy.stats import entropy

def compute_elsa_scores(coords, categories, m_neighbors=15):
    """
    Computes ELSA (Entropy-based local indicators of spatial association)
    using an m-nearest neighbor graph for O(Nm) complexity.
    """
    # 1. Build a KDTree for rapid m-nearest neighbor querying
    tree = cKDTree(coords)
    
    # 2. Query the m-nearest neighbors for every spot (including itself)
    # This directly defines the neighborhood size m
    _, neighbor_indices = tree.query(coords, k=m_neighbors)
    
    elsa_scores = np.zeros(len(coords))
    
    # 3. Compute localized Shannon entropy per neighborhood
    for i, indices in enumerate(neighbor_indices):
        local_categories = categories[indices]
        
        # Calculate the probability distribution within this specific microenvironment
        _, counts = np.unique(local_categories, return_counts=True)
        p_i = counts / m_neighbors
        
        # Calculate entropy (base 2 for bits)
        elsa_scores[i] = entropy(p_i, base=2)
        
    return elsa_scores

# ---------------------------------------------------------
# Execution on HuBMAP Spatial Data
# ---------------------------------------------------------

# Extract spatial coordinates and molecular/cell-type categories
coords = adata.obsm['spatial']
categories = adata.obs['cell_type'].values

# Define neighborhood size (m). 
# Note: Small m misses broader patterns; large m dilutes local signals.
m = 20 

# Compute and store ELSA scores in the AnnData object
adata.obs['ELSA_score'] = compute_elsa_scores(coords, categories, m_neighbors=m)

# Retrieve the previously constructed Graph Laplacian and initial state
L_E = adata.obsp['spatial_laplacian']
f_0 = adata.obs['initial_exerkine_signal'].values

# Execute signal propagation (Heat Diffusion)
t = 0.5
F_t = expm_multiply(-t * L_E, f_0)

# Scale the propagated signal using the ELSA scores
# This weights the signal heavily in heterogeneity hotspots (e.g., immune-cell mixing)
adata.obs['propagated_signal_elsa_scaled'] = F_t * adata.obs['ELSA_score'].values

print("ELSA-scaled signal propagation complete.")

## Constructing Graph Laplacian ($\mathcal{L}_E$) & Simulating Diffusion ($\mathbf{F}(t)$)

In [ ]:
spatial_path = PROCESSED_SPATIAL_DIR / 'motrpac_spatial_propagated.h5ad'
network_path = NETWORK_DIR / 'spatial_communication_network.csv'

if spatial_path.exists() and network_path.exists():
    adata = sc.read_h5ad(spatial_path)
    spatial_network = pd.read_csv(network_path)
    
    # Aggregate edge weights W_E
    edge_weights = spatial_network.groupby(['sender_spot', 'receiver_spot'])['S_tilde_score'].sum().reset_index()
    spot_to_idx = {name: idx for idx, name in enumerate(adata.obs_names)}
    
    row_idx = edge_weights['sender_spot'].map(spot_to_idx).dropna().values.astype(int)
    col_idx = edge_weights['receiver_spot'].map(spot_to_idx).dropna().values.astype(int)
    weights = edge_weights['S_tilde_score'].values[:len(row_idx)]
    
    num_spots = adata.n_obs
    W_E = coo_matrix((weights, (row_idx, col_idx)), shape=(num_spots, num_spots)).tocsr()
    
    # Laplacian L_E = D - W_E
    out_degrees = np.array(W_E.sum(axis=1)).flatten()
    D = diags(out_degrees, format='csr')
    L_E = D - W_E
    
    print(f'Graph Laplacian constructed. Shape: {L_E.shape}, Non-zeros: {L_E.nnz}')
else:
    print('Required spatial network or propagated h5ad dataset not found. Ensure workflow scripts 09 and 10 are executed.')

# 9. UMAP Integration & Multi-Omics Visualization
This notebook projects the integrated high-dimensional cellular molecular embeddings ($\mathbf{Z}$) into 2D UMAP coordinates ($\mathbf{u}_i = \mathcal{U}(\mathbf{z}_i)$) as specified in Section 16 of the [EXERKINEMAP Mathematical Model](https://github.com/gomezdj/exerkinemap/blob/main/mathematical_model.md#16-umap-integration).

## Loading Integrated Spatial AnnData & Generating UMAP Projections

In [ ]:
spatial_path = PROCESSED_SPATIAL_DIR / 'spatial_propagated.h5ad'

if spatial_path.exists():
    adata = sc.read_h5ad(spatial_path)
    
    # Ensure neighbors and UMAP coordinates are computed for integrated multi-omics representations
    if 'neighbors' not in adata.uns:
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    if 'umap' not in adata.obsm:
        sc.tl.umap(adata)
        
    print(f'UMAP coordinates computed. Shape: {adata.obsm["X_umap"].shape}')
    
    # Visualize integrated representations (e.g., pathway activation or propagated signals)
    signal_key = [k for k in adata.obs.keys() if 'propagated_exerkine_signal' in k]
    if signal_key:
        sc.pl.umap(adata, color=signal_key, title='EXERKINEMAP Propagated Signal UMAP')
else:
    print('Propagated spatial dataset not found. Ensure workflow 10 has been executed.')

# 11. Interorgan Exerkine Network Extension
This notebook projects cellular and spatial communication graphs into a systemic interorgan communication graph ($\mathcal{G}_O$) using the tissue mapping function ($\omega: C \rightarrow O$) and aggregates organ-level edge weights ($W_{ab}^O$) as defined in Section 15 of the [EXERKINEMAP Mathematical Model](https://github.com/gomezdj/exerkinemap/blob/main/mathematical_model.md#15-interorgan-extension).

In [ ]:
network_path = NETWORK_DIR / 'spatial_communication_network.csv'
spatial_path = PROCESSED_SPATIAL_DIR / 'motrpac_spatial_propagated.h5ad'

if network_path.exists():
    spatial_network = pd.read_csv(network_path)
    
    # Simulate or map cells to organs (omega: C -> O)
    organs = ['Skeletal Muscle', 'Liver', 'Adipose Tissue', 'Heart', 'Brain']
    spatial_network['sender_organ'] = np.random.choice(organs, size=len(spatial_network))
    spatial_network['receiver_organ'] = np.random.choice(organs, size=len(spatial_network))
    
    # Aggregate interorgan flow matrix: W_ab^O = sum(w_ij^E)
    flow_matrix = spatial_network.groupby(['sender_organ', 'receiver_organ'])['S_tilde_score'].sum().unstack(fill_value=0)
    
    print('Interorgan Flow Matrix (W^O_ab):')
    display(flow_matrix)
else:
    print('Spatial communication network not found. Ensure workflow 09 has been executed.')

# 12. Inverse EXERKINEMAP & Closed-Loop Optimization
This notebook executes the inverse design workflow, generative sequence sampling via ProGen2, and closed-loop loss minimization ($\mathcal{L}_{INV}$) to optimize candidate exercise-responsive exerkine sequences against target pathway activation states ($A_P^*0$).

## 13. Conditional Sequence Generation
Sampling novel protein candidate sequences ($\hat{\mathbf{a}}^P \sim P_{PLM}(\mathbf{a}^P \mid \mathbf{z}^P)$) conditioned on secretory motifs.

In [ ]:
model_name = 'Salesforce/progen2-model'
print(f'Loading generative model: {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(DEVICE)
model.eval()

# Conditioning prompt motif
prompt = 'MKWVTFISLLFLFSSAYSRV'
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        inputs.input_ids,
        max_new_tokens=45,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_seq = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f'Generated Candidate Sequence: {generated_seq}')

## 13. Closed-Loop Optimization Loop
Iteratively minimizing $\mathcal{L}_{INV} = d(\hat{A}_P, A_P^*)$ until convergence.

In [ ]:
def forward_model_simulation(seq):
    np.random.seed(len(seq))
    return np.random.uniform(0.1, 1.0, size=5)

target_a_p = np.array([0.95, 0.90, 0.85, 0.92, 0.88])
current_seq = generated_seq
epsilon = 0.08
max_iter = 5

for iteration in range(1, max_iter + 1):
    a_p_hat = forward_model_simulation(current_seq)
    loss = float(np.mean((a_p_hat - target_a_p) ** 2))
    print(f'Iteration {iteration} | L_INV Loss: {loss:.4f}')
    
    if loss < epsilon:
        print('Convergence criterion satisfied!')
        break
        
    # Simulated sequence optimization mutation
    mutation_idx = np.random.randint(5, len(current_seq))
    amino_acids = 'VLIFAWMCQEDRKHSTY'
    current_seq = current_seq[:mutation_idx] + amino_acids[np.random.randint(0, len(amino_acids))] + current_seq[mutation_idx+1:]

print('Closed-loop optimization pipeline complete.')

## 14. Multimodal Data Map
This script constructs a Multimodal Data Map for the MoTrPAC framework.
It integrates generated sequence data (RNA/Protein), single-cell omics 
(scRNA-seq, snRNA-seq, scATAC-seq), and spatial omics (spatial transcriptomics, 
spatial proteomics) into a unified MuData (Multimodal AnnData) object for downstream 
cross-modal mapping and evaluation.

In [ ]:
import anndata as ad
import mudata as md

def create_directories():
    """Ensure output directories exist."""
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_sequence_data():
    """
    Loads generated exerkine sequences (RNA and Protein).
    Wraps the embeddings (e.g., from Omni-DNA) into an AnnData object.
    """
    logger.info("Loading sequence generation data...")
    # Simulated data mapping for the script structure
    obs = pd.DataFrame(index=[f"seq_{i}" for i in range(100)])
    obs['sequence_type'] = np.random.choice(['RNA', 'Protein'], 100)
    
    # Simulating 512-dimensional sequence embeddings
    X = np.random.rand(100, 512) 
    
    return ad.AnnData(X=X, obs=obs)

def load_single_cell_data():
    """
    Loads single-cell omics (scRNA-seq, snRNA-seq, scATAC-seq).
    """
    logger.info("Loading single-cell omics data...")
    obs = pd.DataFrame(index=[f"cell_{i}" for i in range(500)])
    obs['cell_type'] = np.random.choice(['Myocyte', 'Fibroblast', 'Macrophage'], 500)
    obs['assay_type'] = np.random.choice(['scRNAseq', 'snRNAseq', 'scATACseq'], 500)
    
    # Simulating 2000 genetic/epigenetic features
    X = np.random.rand(500, 2000) 
    
    return ad.AnnData(X=X, obs=obs)

def load_spatial_data():
    """
    Loads spatial omics (spatial transcriptomics, spatial proteomics).
    Includes physical coordinate mapping in the .obsm layer.
    """
    logger.info("Loading spatial omics data...")
    obs = pd.DataFrame(index=[f"spot_{i}" for i in range(300)])
    obs['tissue_region'] = np.random.choice(['Muscle_Fiber', 'Interstitium', 'Vascular'], 300)
    obs['assay_type'] = np.random.choice(['Spatial_Transcriptomics', 'Spatial_Proteomics'], 300)
    
    # Simulating 1500 spatial features
    X = np.random.rand(300, 1500)
    
    # Spatial coordinates mapping (X, Y)
    spatial_coords = np.random.rand(300, 2) * 100
    obsm = {"spatial": spatial_coords}
    
    return ad.AnnData(X=X, obs=obs, obsm=obsm)

def build_multimodal_map(seq_ad, sc_ad, st_ad):
    """
    Constructs a MuData object containing all independent omics modalities.
    """
    logger.info("Constructing the unified Multimodal Data Map...")
    
    # Dictionary mapping for the MuData structure
    modalities = {
        "sequences": seq_ad,
        "single_cell": sc_ad,
        "spatial": st_ad
    }
    
    # Initialize Multimodal AnnData
    mdata = md.MuData(modalities)
    return mdata

def main():
    logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
    logger.info("Initializing Phase 15: Multimodal Data Map for the MoTrPAC framework...")
    create_directories()

    # 1. Load diverse data modalities
    seq_ad = load_sequence_data()
    sc_ad = load_single_cell_data()
    st_ad = load_spatial_data()

    # 2. Build the Multimodal Map
    mdata = build_multimodal_map(seq_ad, sc_ad, st_ad)
    
    # 3. Save the integrated multimodal dataset
    output_path = RESULTS_DIR / "motrpac_multimodal_map.h5mu"
    mdata.write(output_path)
    
    logger.info(f"Multimodal Data Map successfully saved to {output_path}")
    logger.info(f"MuData Map Structure:\n{mdata}")

if __name__ == "__main__":
    main()

In [ ]:
#%% Load Sequence Generation Data
def load_sequence_data():
    """Loads generated exerkine sequences (RNA and Protein) into an AnnData object."""
    logger.info("Loading sequence generation data...")
    obs = pd.DataFrame(index=[f"seq_{i}" for i in range(100)])
    obs['sequence_type'] = np.random.choice(['RNA', 'Protein'], 100)
    
    # Simulating 512-dimensional sequence embeddings
    X = np.random.rand(100, 512) 
    return ad.AnnData(X=X, obs=obs)

seq_ad = load_sequence_data()
display(seq_ad)

#%% Load Single-Cell Omics Data
def load_single_cell_data():
    """Loads single-cell omics data (scRNA-seq, snRNA-seq, scATAC-seq)."""
    logger.info("Loading single-cell omics data...")
    obs = pd.DataFrame(index=[f"cell_{i}" for i in range(500)])
    obs['cell_type'] = np.random.choice(['Myocyte', 'Fibroblast', 'Macrophage'], 500)
    obs['assay_type'] = np.random.choice(['scRNAseq', 'snRNAseq', 'scATACseq'], 500)
    
    # Simulating 2000 genetic/epigenetic features
    X = np.random.rand(500, 2000) 
    return ad.AnnData(X=X, obs=obs)

sc_ad = load_single_cell_data()
display(sc_ad)

#%% Load Spatial Omics Data
def load_spatial_data():
    """Loads spatial omics data with physical coordinate mapping."""
    logger.info("Loading spatial omics data...")
    obs = pd.DataFrame(index=[f"spot_{i}" for i in range(300)])
    obs['tissue_region'] = np.random.choice(['Muscle_Fiber', 'Interstitium', 'Vascular'], 300)
    obs['assay_type'] = np.random.choice(['Spatial_Transcriptomics', 'Spatial_Proteomics'], 300)
    
    # Simulating 1500 spatial features and 2D spatial coordinates
    X = np.random.rand(300, 1500)
    spatial_coords = np.random.rand(300, 2) * 100
    obsm = {"spatial": spatial_coords}
    
    return ad.AnnData(X=X, obs=obs, obsm=obsm)

st_ad = load_spatial_data()
display(st_ad)

#%% Build and Save the Multimodal Data Map
def build_multimodal_map(seq_ad, sc_ad, st_ad):
    """Constructs a MuData object containing all independent omics modalities."""
    logger.info("Constructing the unified Multimodal Data Map...")
    modalities = {
        "sequences": seq_ad,
        "single_cell": sc_ad,
        "spatial": st_ad
    }
    return md.MuData(modalities)

mdata = build_multimodal_map(seq_ad, sc_ad, st_ad)

# Save to disk
output_path = RESULTS_DIR / "motrpac_multimodal_map.h5mu"
mdata.write(output_path)

logger.info(f"Multimodal Data Map successfully saved to {output_path}")
display(mdata)

# 15. Summary 

This code block provides a comprehensive, programmatic summary of the entire tutorial's analytical foundation by organizing the information into two structured tables: EXERKINEMAP (MM) Summary: Outlines the core quantitative components of the framework, including Generalized Linear Models ($g(\mu) = X\beta + \epsilon$) for feature mapping, the sequential forward model pipeline ($\hat{A}_0 \to \hat{G}_E \to \hat{F}(t) \to \hat{A}_P$), and the inverse design closed-loop loss function ($\mathcal{L}_{INV}$ using MSE).Generated Data Inventory: Catalogs the various datasets, formats, and structural artifacts produced across the pipeline—ranging from sequence reference libraries and single-cell/spatial AnnData matrices to unified MuData (.h5mu) objects, network topologies, and final optimization reports (.csv, .png, .md).The script uses Pandas dataframes to display these summaries cleanly inside a Jupyter Notebook environment.

In [ ]:
#%% [markdown]
# # Mathematical Model and Data Summary
# This notebook provides an automated summary of the core mathematical framework and the data modalities generated across the MoTrPAC tutorial pipeline.

#%% Import Dependencies
import logging
import pandas as pd
from pathlib import Path

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

#%% Define Mathematical Model Summary
def get_math_model_summary():
    """Returns a structured summary of the core mathematical components."""
    components = [
        {
            "Component": "Generalized Linear Models (GLMs)",
            "Equation / Form": "g(μ) = Xβ + ε",
            "Description": "Relates multi-omics and spatial features (X) to functional pathway annotations (μ) via link functions."
        },
        {
            "Component": "Forward Model Simulation",
            "Equation / Form": "Â₀ → Ĝ_E → F̂(t) → Â_P",
            "Description": "Maps candidate sequences through structural embeddings and dynamic signal propagation to predict pathway activation vectors."
        },
        {
            "Component": "Inverse Design & Closed-Loop Loss",
            "Equation / Form": "L_INV = (1/n) Σ (Â_P,i - A_P*,i)²",
            "Description": "Minimizes Mean Squared Error between predicted pathway activation and the target state (A_P*) via iterative mutation."
        }
    ]
    return pd.DataFrame(components)

#%% Define Data Generation Inventory
def get_generated_data_inventory():
    """Returns a catalog of the datasets and artifacts produced by the tutorial."""
    data = [
        {"Modality / Asset", "Format / File Type", "Description"},
        {"Sequence Reference Libraries", "FASTA / CSV", "Extracted and generated RNA and protein sequences for exerkine ligands and receptors."},
        {"Single-Cell & Spatial Omics", "AnnData (.h5ad)", "Processed scRNA-seq, snRNA-seq, scATAC-seq, spatial transcriptomics, and spatial proteomics matrices with spatial coordinates."},
        {"Multimodal Data Maps", "MuData (.h5mu)", "Unified cross-modal data structures integrating sequences, single-cell cellular states, and tissue regions."},
        {"Network Topologies", "Graph / CSV", "Ligand-receptor interaction networks, intracellular signaling paths, and cross-organ system maps."},
        {"Optimization Summaries", "CSV / PNG / Markdown", "Candidate convergence metrics, final inverse loss values (L_INV), and automated performance reports."}
    ]
    # Convert list of lists/tuples to DataFrame (using first row as header)
    df = pd.DataFrame(data[1:], columns=data[0])
    return df

#%% Execute and Display Summaries
math_df = get_math_model_summary()
data_df = get_generated_data_inventory()

logger.info("=== Core Mathematical Model Components ===")
display(math_df)

logger.info("\n=== Generated Data Modalities & Assets ===")
display(data_df)